In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Milestone 3 — Retrieval-Augmented Generation (RAG)

This notebook follows the Milestone 3 workflow:

1. Build a knowledge base from the correct answer text in `train.csv`.
2. Create a FAISS vector index using `all-MiniLM-L6-v2`.
3. Run zero-shot classification using `facebook/bart-large-mnli`.
4. Retrieve documents with FAISS.
5. Re-rank retrieved documents using a Cross-Encoder.
6. Compare zero-shot, good-context RAG, and adversarial RAG.
7. Calculate Hit Rate and MAP@3.

> Run the notebook cells in order.

In [1]:
!pip install -q faiss-cpu sentence-transformers transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 42.5 MB/s eta 0:00:0000:0100:01


In [2]:
import os
import numpy as np
import pandas as pd
import faiss
import torch

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, pipeline

np.random.seed(42)
torch.manual_seed(42)

possible_paths = [
    "train.csv",
    "/kaggle/working/train.csv",
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv",
]

train_path = next((path for path in possible_paths if os.path.exists(path)), None)

if train_path is None:
    for root, _, files in os.walk("/kaggle/input"):
        if "train.csv" in files:
            train_path = os.path.join(root, "train.csv")
            break

if train_path is None:
    raise FileNotFoundError(
        "train.csv was not found. Upload it beside the notebook or attach the competition dataset."
    )

train = pd.read_csv(train_path)
print("Loaded:", train_path)
print("Shape:", train.shape)
train.head()

Loaded: /kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
Shape: (2000, 8)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


# Milestone 3 Setup

The knowledge base contains the exact text of the correct option for every training row.

FAISS uses embeddings from `all-MiniLM-L6-v2` to retrieve semantically similar documents.

In [3]:
print("Creating knowledge base")

kb = []
for _, row in train.iterrows():
    correct_letter = row["answer"]
    kb.append(str(row[correct_letter]))

print("Loading embedding model and creating FAISS index")

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
kb_embeddings = embedding_model.encode(
    kb,
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

index = faiss.IndexFlatL2(kb_embeddings.shape[1])
index.add(kb_embeddings)

print("Knowledge base successfully created")
print("Documents in KB:", len(kb))
print("Embedding dimension:", kb_embeddings.shape[1])

Creating knowledge base
Loading embedding model and creating FAISS index


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Knowledge base successfully created
Documents in KB: 2000
Embedding dimension: 384


In [4]:
zs = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0 if torch.cuda.is_available() else -1
)

cross_encoder = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device="cuda" if torch.cuda.is_available() else "cpu"
)

bert_tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

CHOICE_LETTERS = ["A", "B", "C", "D", "E"]

def get_options(row):
    return [str(row[letter]) for letter in CHOICE_LETTERS]

def retrieve(prompt, k=5):
    query_embedding = embedding_model.encode(
        [str(prompt)],
        show_progress_bar=False,
        convert_to_numpy=True
    ).astype("float32")
    distances, indices = index.search(query_embedding, k)
    return distances[0], indices[0]

def probability_for_correct_option(result, correct_text):
    for label, score in zip(result["labels"], result["scores"]):
        if label == correct_text:
            return float(score)
    raise ValueError("Correct option text was not found in classifier output.")

def map_at_3(predicted_letters, correct_letter):
    if correct_letter not in predicted_letters[:3]:
        return 0.0
    rank = predicted_lette

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

# Q1. Zero-Shot Probability for Row 150

Run `facebook/bart-large-mnli` on the prompt at row index 150 using its five answer options as candidate labels.

The requested value is the probability assigned to the ground-truth correct option.

In [5]:
row_150 = train.iloc[150]
prompt_150 = str(row_150["prompt"])
labels_150 = get_options(row_150)
correct_letter_150 = str(row_150["answer"])
ans_150 = str(row_150[correct_letter_150])

q1_result = zs(prompt_150, candidate_labels=labels_150, multi_label=False)
q1_score = probability_for_correct_option(q1_result, ans_150)

print("Correct letter:", correct_letter_150)
print("Correct option:", ans_150)
print("Probability assigned to correct option:", round(q1_score, 3))
print("Expected answer: 0.384")

Correct letter: C
Correct option: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Probability assigned to correct option: 0.384
Expected answer: 0.384


# Q2. FAISS Rank of the True Document

Embed the prompt at row index 150 and retrieve the top 10 documents.

Find the rank of KB document index `150` within those results.

In [6]:
q2_distances, retrieved_indices = retrieve(prompt_150, k=10)

positions = np.where(retrieved_indices == 150)[0]
true_faiss_rank = int(positions[0] + 1) if len(positions) else None

print("Retrieved KB indices:", retrieved_indices.tolist())
print("FAISS rank of KB index 150:", true_faiss_rank)
print("Expected answer: 10")

Retrieved KB indices: [663, 1701, 1269, 1532, 576, 847, 1693, 1906, 168, 150]
FAISS rank of KB index 150: 10
Expected answer: 10


# Q3. Cross-Encoder Re-Ranking

Take the ten documents retrieved in Q2 and score each `(prompt, document)` pair using `cross-encoder/ms-marco-MiniLM-L-6-v2`.

Sort by descending score and find the new rank of KB index `150`.

In [8]:
docs_10 = [kb[i] for i in retrieved_indices]
pairs_10 = [[prompt_150, doc] for doc in docs_10]

ce_scores_10 = cross_encoder.predict(pairs_10)
rerank_order = np.argsort(ce_scores_10)[::-1]

reranked_indices = retrieved_indices[rerank_order]
reranked_scores = np.asarray(ce_scores_10)[rerank_order]

positions = np.where(reranked_indices == 150)[0]
true_cross_encoder_rank = int(positions[0] + 1) if len(positions) else None

rerank_table = pd.DataFrame({
    "reranked_position": np.arange(1, len(reranked_indices) + 1),
    "kb_index": reranked_indices,
    "cross_encoder_score": reranked_scores,
    "document": [kb[i] for i in reranked_indices]
})

display(rerank_table)

print("Cross-Encoder rank of KB index 150:", true_cross_encoder_rank)
print("Expected answer: 1")

,reranked_position,kb_index,cross_encoder_score,document
0,1,150,4.758512,The butterfly effect is the phenomenon that a ...
1,2,1906,4.752576,The butterfly effect is the phenomenon that a ...
2,3,847,4.752576,The butterfly effect is the phenomenon that a ...
3,4,1693,4.752576,The butterfly effect is the phenomenon that a ...
4,5,1269,4.737523,The butterfly effect is the phenomenon that a ...
5,6,1532,4.737523,The butterfly effect is the phenomenon that a ...
6,7,168,4.707248,The butterfly effect is the phenomenon that a ...
7,8,576,4.687045,The butterfly effect is the phenomenon that a ...
8,9,1701,4.660232,The butterfly effect is the phenomenon that a ...
9,10,663,4.660232,The butterfly effect is the phenomenon that a ...


Cross-Encoder rank of KB index 150: 1
Expected answer: 1


# Q4. Token Count for a Five-Document RAG Prompt

For row index 42, retrieve the top five documents, join them with one space, create exactly:

`Context: [concatenated_docs] Question: [prompt]`

Then tokenize using `bert-base-uncased` without truncation.

In [10]:
row_42 = train.iloc[42]
prompt_42 = str(row_42["prompt"])

_, indices_42 = retrieve(prompt_42, k=5)
docs_42 = [kb[i] for i in indices_42]
concatenated_docs_42 = " ".join(docs_42)

rag_string_42 = f"Context: {concatenated_docs_42} Question: {prompt_42}"
tokens_42 = bert_tokenizer(rag_string_42, truncation=False, add_special_tokens=True)

q4_token_count = len(tokens_42["input_ids"])

print("Retrieved KB indices:", indices_42.tolist())
print("Total tokens:", q4_token_count)
print("Expected answer: 216")

Retrieved KB indices: [241, 439, 456, 506, 605]
Total tokens: 216
Expected answer: 216


# Q5. RAG with the Exact True Document

Use the exact KB document at index `150` as context, then run the same zero-shot classifier.

In [11]:
true_document_150 = kb[150]
rag_string_true = f"Context: {true_document_150} Question: {prompt_150}"

q5_result = zs(rag_string_true, candidate_labels=labels_150, multi_label=False)
q5_score = probability_for_correct_option(q5_result, ans_150)

print("True document:", true_document_150)
print("Probability assigned to correct option:", round(q5_score, 3))
print("Expected answer: 0.989")

True document: The butterfly effect is the phenomenon that a small change in the initial conditions of a dynamical framework can cause significant distinctions in subsequent states, as defined by Lorenz in his book "The Essence of Chaos."
Probability assigned to correct option: 0.989
Expected answer: 0.989


# Q6. Adversarial RAG with Incorrect Context

Force the unrelated KB document at index `999` into the context for row `150`.

In [12]:
wrong_document_999 = kb[999]
adversarial_rag_string = f"Context: {wrong_document_999} Question: {prompt_150}"

q6_result = zs(adversarial_rag_string, candidate_labels=labels_150, multi_label=False)
q6_score = probability_for_correct_option(q6_result, ans_150)

print("Forced KB document index: 999")
print("Incorrect context:", wrong_document_999)
print("Probability assigned to correct option:", round(q6_score, 3))
print("Expected answer: 0.529")

Forced KB document index: 999
Incorrect context: A thought experiment in which a demon guards a microscopic trapdoor in a wall separating two parts of a container filled with the same gas at equal temperatures. The demon selectively allows faster-than-average molecules to pass from one side to the other, causing a reduce in temperature in one part and an boost in temperature in the other, contrary to the second law of thermodynamics.
Probability assigned to correct option: 0.529
Expected answer: 0.529


# Q7. Top-5 Retrieval Hit Rate

For the first 100 rows, count a hit when the exact correct-option string appears inside any of the five retrieved documents.

In [13]:
hits = []
hit_details = []

for row_index in range(100):
    row = train.iloc[row_index]
    prompt = str(row["prompt"])
    correct_text = str(row[row["answer"]])

    _, retrieved = retrieve(prompt, k=5)
    retrieved_docs = [kb[i] for i in retrieved]

    is_hit = any(correct_text in doc for doc in retrieved_docs)
    hits.append(int(is_hit))

    hit_details.append({
        "row_index": row_index,
        "correct_text": correct_text,
        "retrieved_indices": retrieved.tolist(),
        "hit": is_hit
    })

q7_hit_rate = 100 * np.mean(hits)

print("Number of hits:", sum(hits), "out of", len(hits))
print("Hit Rate (%):", round(q7_hit_rate, 1))
print("Expected answer: 73.0")

Number of hits: 73 out of 100
Hit Rate (%): 73.0
Expected answer: 73.0


# Q8. Complete Retrieve → Re-Rank → Augment → Predict Pipeline

Process rows `0` through `19`, calculate AP@3 for each row, and average the values to obtain MAP@3.

In [18]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import pipeline

if "train" not in globals():
    train = pd.read_csv("train.csv")

if "kb" not in globals():
    kb = []
    for _, row in train.iterrows():
        correct_letter = str(row["answer"]).strip()
        kb.append(str(row[correct_letter]))

if "model" not in globals():
    model = SentenceTransformer("all-MiniLM-L6-v2")

if "index" not in globals():
    kb_embeddings = model.encode(
        kb,
        show_progress_bar=True,
        convert_to_numpy=True
    ).astype("float32")

    index = faiss.IndexFlatL2(kb_embeddings.shape[1])
    index.add(kb_embeddings)

if "cross_encoder" not in globals():
    cross_encoder = CrossEncoder(
        "cross-encoder/ms-marco-MiniLM-L-6-v2"
    )

if "zs" not in globals():
    zs = pipeline(
        "zero-shot-classification",
        model="facebook/bart-large-mnli"
    )

def map_at_3(predicted_letters, correct_letter):
    top_3 = predicted_letters[:3]

    if correct_letter not in top_3:
        return 0.0

    rank = top_3.index(correct_letter) + 1
    return 1.0 / rank

q8_rows = []
map3_scores = []

option_letters = ["A", "B", "C", "D", "E"]

for row_index in range(20):
    row = train.iloc[row_index]

    prompt = str(row["prompt"])
    correct_letter = str(row["answer"]).strip()

    candidate_labels = [
        str(row[letter])
        for letter in option_letters
    ]

    prompt_embedding = model.encode(
        [prompt],
        show_progress_bar=False,
        convert_to_numpy=True
    ).astype("float32")

    distances, retrieved_indices_array = index.search(
        prompt_embedding,
        5
    )

    retrieved_indices = retrieved_indices_array[0].tolist()

    retrieved_documents = [
        kb[document_index]
        for document_index in retrieved_indices
    ]

    prompt_document_pairs = [
        [prompt, document]
        for document in retrieved_documents
    ]

    cross_encoder_scores = cross_encoder.predict(
        prompt_document_pairs,
        show_progress_bar=False
    )

    best_position = int(np.argmax(cross_encoder_scores))

    best_document = retrieved_documents[best_position]
    best_document_index = retrieved_indices[best_position]

    rag_string = f"Context: {best_document} Question: {prompt}"

    prediction = zs(
        rag_string,
        candidate_labels=candidate_labels,
        multi_label=False
    )

    ranked_option_texts = prediction["labels"]

    predicted_letters = []

    for predicted_text in ranked_option_texts:
        for letter, option_text in zip(
            option_letters,
            candidate_labels
        ):
            if predicted_text == option_text:
                predicted_letters.append(letter)
                break

    top_3_letters = predicted_letters[:3]

    row_ap3 = map_at_3(
        top_3_letters,
        correct_letter
    )

    map3_scores.append(row_ap3)

    q8_rows.append({
        "row_index": row_index,
        "correct_answer": correct_letter,
        "top_3_predictions": top_3_letters,
        "map_at_3": row_ap3,
        "retrieved_indices": retrieved_indices,
        "best_document_index": best_document_index,
        "best_document": best_document
    })

    print(
        f"Row {row_index:02d} | "
        f"Correct: {correct_letter} | "
        f"Top 3: {top_3_letters} | "
        f"AP@3: {row_ap3:.3f}"
    )

q8_results = pd.DataFrame(q8_rows)

final_map_at_3 = float(np.mean(map3_scores))

print("\n" + "=" * 60)
print(f"Final average MAP@3: {final_map_at_3:.3f}")
print("=" * 60)

q8_results

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Row 00 | Correct: B | Top 3: ['B', 'D', 'A'] | AP@3: 1.000
Row 01 | Correct: A | Top 3: ['A', 'E', 'C'] | AP@3: 1.000
Row 02 | Correct: C | Top 3: ['C', 'D', 'B'] | AP@3: 1.000
Row 03 | Correct: B | Top 3: ['B', 'D', 'A'] | AP@3: 1.000
Row 04 | Correct: A | Top 3: ['A', 'B', 'C'] | AP@3: 1.000
Row 05 | Correct: C | Top 3: ['B', 'C', 'A'] | AP@3: 0.500
Row 06 | Correct: E | Top 3: ['E', 'B', 'D'] | AP@3: 1.000
Row 07 | Correct: A | Top 3: ['A', 'B', 'C'] | AP@3: 1.000
Row 08 | Correct: A | Top 3: ['A', 'C', 'D'] | AP@3: 1.000
Row 09 | Correct: A | Top 3: ['A', 'B', 'C'] | AP@3: 1.000
Row 10 | Correct: C | Top 3: ['C', 'A', 'D'] | AP@3: 1.000
Row 11 | Correct: B | Top 3: ['B', 'C', 'A'] | AP@3: 1.000
Row 12 | Correct: D | Top 3: ['D', 'A', 'B'] | AP@3: 1.000
Row 13 | Correct: E | Top 3: ['E', 'D', 'B'] | AP@3: 1.000
Row 14 | Correct: E | Top 3: ['E', 'A', 'D'] | AP@3: 1.000
Row 15 | Correct: E | Top 3: ['E', 'B', 'C'] | AP@3: 1.000
Row 16 | Correct: C | Top 3: ['C', 'B', 'E'] | AP@3: 1.0

,row_index,correct_answer,top_3_predictions,map_at_3,retrieved_indices,best_document_index,best_document
0,0,B,"[B, D, A]",1.0,"[0, 3, 71, 139, 560]",0,Martin Heidegger believes that humans do not e...
1,1,A,"[A, E, C]",1.0,"[140, 699, 931, 1028, 919]",140,Accelerator-based light-ion fusion is a techni...
2,2,C,"[C, D, B]",1.0,"[743, 1540, 501, 368, 902]",368,The matter and radiation that exist in the spa...
3,3,B,"[B, D, A]",1.0,"[0, 3, 71, 139, 560]",0,Martin Heidegger believes that humans do not e...
4,4,A,"[A, B, C]",1.0,"[4, 292, 383, 499, 595]",4,"Simultaneity is relative, meaning that two eve..."
5,5,C,"[B, C, A]",0.5,"[249, 1109, 1165, 1515, 1590]",249,The Josephson effect is a phenomenon exploited...
6,6,E,"[E, B, D]",1.0,"[6, 15, 245, 508, 909]",6,A pattern left by a particle-laden liquid afte...
7,7,A,"[A, B, C]",1.0,"[7, 47, 69, 119, 733]",7,The Liouville density is a probability distrib...
8,8,A,"[A, C, D]",1.0,"[255, 288, 367, 384, 515]",255,Distinguishing background particles from dark ...
9,9,A,"[A, B, C]",1.0,"[9, 81, 305, 753, 881]",9,"The Wigner function W(x, p) is the Wigner tran..."
